# DriveSense — 01. Data Acquisition & EDA

**Status: not yet executed against real data.** This notebook is written to run in a clean Google Colab runtime, per the project's Colab-first reproducibility requirement (see `docs/Rubric_Alignment.md`).

It downloads the two source datasets (see `docs/Dataset_Research.md`) and runs exploratory data analysis: class balance, subject counts (for subject-independent splitting), sample images, and basic data-quality checks.

No numbers, plots, or findings below are real until this notebook has actually been run — do not copy placeholder structure into the README/Brief as if it were a result.

## 0. Setup

In [ ]:
# Run once per Colab session.
!git clone https://github.com/echo05w/drivesense-driver-monitoring.git
%cd drivesense-driver-monitoring
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
# Upload your own kaggle.json (Kaggle account -> Settings -> Create New API Token).
# This must be the STUDENT'S OWN Kaggle credentials, never committed to the repo.
from google.colab import files
uploaded = files.upload()  # select kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

## 1. Download datasets

In [ ]:
!python scripts/download_data.py --dataset distraction
!python scripts/download_data.py --dataset drowsiness

## 2. Distraction dataset (State Farm) EDA

Checks: class distribution, subject (driver) count and per-subject sample count (for subject-independent split sizing), sample image grid, image size/corruption check.

In [ ]:
import pandas as pd
from pathlib import Path

DISTRACTION_DIR = Path('data/raw/distraction')
labels_csv = DISTRACTION_DIR / 'driver_imgs_list.csv'
distraction_df = pd.read_csv(labels_csv)
print(distraction_df.shape)
distraction_df.head()

In [ ]:
import matplotlib.pyplot as plt

class_counts = distraction_df['classname'].value_counts().sort_index()
class_counts.plot(kind='bar', title='Distraction class distribution (train)')
plt.ylabel('image count')
plt.show()

subject_counts = distraction_df['subject'].value_counts()
print(f"Number of unique subjects: {distraction_df['subject'].nunique()}")
subject_counts.plot(kind='bar', figsize=(12, 4), title='Images per subject')
plt.show()

In [ ]:
# Sample image grid — one random image per class, for a visual sanity check.
import cv2

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for ax, (classname, group) in zip(axes.flat, distraction_df.groupby('classname')):
    row = group.sample(1, random_state=0).iloc[0]
    img_path = DISTRACTION_DIR / 'imgs' / 'train' / row['classname'] / row['img']
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(classname, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Data-quality check: verify every referenced image file actually exists and opens.
missing, corrupt = [], []
for _, row in distraction_df.iterrows():
    img_path = DISTRACTION_DIR / 'imgs' / 'train' / row['classname'] / row['img']
    if not img_path.exists():
        missing.append(str(img_path))
        continue
    img = cv2.imread(str(img_path))
    if img is None:
        corrupt.append(str(img_path))

print(f"Missing files: {len(missing)}")
print(f"Corrupt/unreadable files: {len(corrupt)}")

## 3. Drowsiness dataset (UTA-RLDD) EDA

Checks: subject count, video count/duration per class, frame-extraction sanity check. Note the full dataset is large (~111 GB raw per `docs/Dataset_Research.md`) — this section should work off the Kaggle mirror subset actually downloaded, and must document if only a subsample of subjects was used.

In [ ]:
DROWSINESS_DIR = Path('data/raw/drowsiness')
video_paths = list(DROWSINESS_DIR.rglob('*.mp4')) + list(DROWSINESS_DIR.rglob('*.mov')) + list(DROWSINESS_DIR.rglob('*.avi'))
print(f"Video files found: {len(video_paths)}")
for p in video_paths[:10]:
    print(p)

In [ ]:
# TODO once real files are downloaded and their naming/labeling scheme is confirmed:
# - parse subject ID and class (alert / low_vigilant / drowsy) from path or a metadata file
# - build a dataframe with columns [subject_id, video_path, label, duration_sec]
# - plot class balance and per-subject video count, same as the distraction section above
# - sample and display a few extracted frames per class for a visual sanity check
raise NotImplementedError(
    'Fill in once the drowsiness dataset has actually been downloaded and its '
    'real file/label layout is confirmed -- do not guess a schema and report '
    'fake EDA numbers.'
)

## 4. Findings to carry into `docs/Rubric_Alignment.md` and `docs/Responsible_AI.md`

After running the cells above for real, transcribe here (and back into the docs):
- Actual class balance for both tasks (and whether resampling/class-weighting is needed)
- Actual subject counts (confirms whether subject-independent split sizes in the Brief are realistic)
- Any missing/corrupt files found
- Visual observations relevant to Responsible AI (e.g., apparent lack of diversity in lighting, eyewear, camera angle)